# go-gost — SOCKS5/HTTP Proxy Setup

go-gost v3 supports SOCKS5, HTTP, TCP/UDP forwarding, proxy chaining, TLS, SSH, and more — all from a single YAML config.

## Generate a SOCKS5 config

In [ ]:
import yaml, pathlib

config = {
    "log": {"level": "info"},
    "services": [
        {
            "name": "socks5",
            "addr": ":1080",
            "handler": {"type": "socks5"},
            "listener": {"type": "tcp"},
        },
        {
            "name": "http-proxy",
            "addr": ":8080",
            "handler": {"type": "http"},
            "listener": {"type": "tcp"},
        },
    ],
}

out = pathlib.Path("/tmp/gost-config.yaml")
out.write_text(yaml.dump(config, default_flow_style=False))
print(out.read_text())

## Validate with yamllint

In [ ]:
import subprocess, sys

result = subprocess.run(
    ["yamllint", "-d", "{extends: relaxed, rules: {line-length: {max: 120}}}", "/tmp/gost-config.yaml"],
    capture_output=True, text=True
)
print(result.stdout or "No issues found.")
if result.returncode != 0:
    print(result.stderr)

## Run go-gost via Docker

In [ ]:
docker_cmd = """
docker run --rm -d \\
  --name gost-test \\
  -p 1080:1080 \\
  -p 8080:8080 \\
  -v /tmp/gost-config.yaml:/etc/gost/config.yaml:ro \\
  gogost/gost:latest -C /etc/gost/config.yaml
""".strip()
print("Run this to start go-gost:")
print(docker_cmd)

## Test the SOCKS5 proxy

In [ ]:
test_cmd = "curl -s --proxy socks5h://localhost:1080 https://api.ipify.org"
print("Test command:")
print(test_cmd)
print()
print("Expected: your public IP address printed from the proxy's perspective")

## Proxy Chain Example

In [ ]:
chain_config = {
    "log": {"level": "info"},
    "services": [
        {
            "name": "entry",
            "addr": ":8080",
            "handler": {"type": "http+socks5"},
            "listener": {"type": "tcp"},
            "chain": "two-hop",
        }
    ],
    "chains": [
        {
            "name": "two-hop",
            "hops": [
                {"name": "hop1", "nodes": [{"name": "proxy1", "addr": "proxy1.example.com:1080",
                                            "connector": {"type": "socks5"}}]},
                {"name": "hop2", "nodes": [{"name": "proxy2", "addr": "proxy2.example.com:8080",
                                            "connector": {"type": "http"}}]},
            ],
        }
    ],
}
print(yaml.dump(chain_config, default_flow_style=False))